In [1]:
import pandas as pd

apercu = pd.read_csv("2013.csv", nrows=5)

print("Nombre total de colonnes :", apercu.shape[1])
display(apercu.iloc[:, :8])

PermissionError: [Errno 13] Permission denied: '2013.csv'

In [2]:
from pathlib import Path

chemin = Path("2013.csv")
print("Chemin utilisé :", chemin.resolve())
print("C'est un dossier :", chemin.is_dir())
print("C'est un fichier :", chemin.is_file())

Chemin utilisé : C:\Users\Fatim\OneDrive\Desktop\Emsi\Machine learning\projet\2013.csv
C'est un dossier : True
C'est un fichier : False


In [3]:
from pathlib import Path

dossier = Path("2013.csv")

for fichier in dossier.iterdir():
    print(fichier.name)

2013.csv


In [4]:
import pandas as pd

apercu = pd.read_csv("2013.csv/2013.csv", nrows=5)

print("Nombre total de colonnes :", apercu.shape[1])
display(apercu.iloc[:, :8])

Nombre total de colonnes : 336


,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO
0,1.0,1.0,b'01092013',b'01',b'09',b'2013',1100.0,2.013001e+09
1,1.0,1.0,b'01192013',b'01',b'19',b'2013',1100.0,2.013001e+09
2,1.0,1.0,b'01192013',b'01',b'19',b'2013',1100.0,2.013001e+09
3,1.0,1.0,b'01112013',b'01',b'11',b'2013',1100.0,2.013001e+09
4,1.0,2.0,b'02062013',b'02',b'06',b'2013',1100.0,2.013001e+09


In [5]:
print("Nombre total de colonnes :", apercu.shape[1])

print("\nLes 30 premières colonnes :")
for colonne in apercu.columns[:30]:
    print(colonne)

Nombre total de colonnes : 336

Les 30 premières colonnes :
_STATE
FMONTH
IDATE
IMONTH
IDAY
IYEAR
DISPCODE
SEQNO
_PSU
CTELENUM
PVTRESD1
COLGHOUS
STATERES
CELLFON3
LADULT
NUMADULT
NUMMEN
NUMWOMEN
GENHLTH
PHYSHLTH
MENTHLTH
POORHLTH
HLTHPLN1
PERSDOC2
MEDCOST
CHECKUP1
SLEPTIM1
BPHIGH4
BPMEDS
BLOODCHO


In [6]:
import pandas as pd

cible = pd.read_csv("2013.csv/2013.csv", usecols=["DIABETE3"])

effectifs = cible["DIABETE3"].value_counts(dropna=False).sort_index()
display(effectifs.to_frame("Nombre de réponses"))

,Nombre de réponses
1.0,62345
2.0,4607
3.0,415394
4.0,8596
7.0,581
9.0,243
NaN,7


In [7]:
cible_nettoyee = cible.loc[cible["DIABETE3"].isin([1, 3])].copy()
cible_nettoyee["diabete"] = cible_nettoyee["DIABETE3"].map({1: 1, 3: 0})

print("Nombre de lignes gardées :", len(cible_nettoyee))
display(cible_nettoyee["diabete"].value_counts().sort_index())

Nombre de lignes gardées : 477739


0    415394
1     62345
Name: diabete, dtype: int64

In [8]:
variables = ["_AGEG5YR", "_BMI5", "EXERANY2"]

donnees = pd.read_csv("2013.csv/2013.csv", usecols=variables)
donnees = donnees.loc[cible_nettoyee.index].copy()
donnees["diabete"] = cible_nettoyee["diabete"]

print("Dimensions :", donnees.shape)
display(donnees.head())

Dimensions : (477739, 4)


,EXERANY2,_AGEG5YR,_BMI5,diabete
0,2.0,9.0,3916.0,0
1,1.0,7.0,1822.0,0
2,2.0,8.0,2746.0,0
3,1.0,9.0,2197.0,0
4,2.0,10.0,3594.0,0


In [9]:
print("Valeurs manquantes par colonne :")
display(donnees.isna().sum().to_frame("Nombre de NaN"))

print("Réponses pour l’activité physique :")
display(donnees["EXERANY2"].value_counts(dropna=False).sort_index())

print("Codes des tranches d’âge :")
display(donnees["_AGEG5YR"].value_counts(dropna=False).sort_index())

print("Résumé de l’IMC brut :")
display(donnees["_BMI5"].describe())

Valeurs manquantes par colonne :


,Nombre de NaN
EXERANY2,30537
_AGEG5YR,0
_BMI5,25620
diabete,0


Réponses pour l’activité physique :


1.0    323392
2.0    121253
7.0       531
9.0      2026
NaN     30537
Name: EXERANY2, dtype: int64

Codes des tranches d’âge :


1.0     26816
2.0     22394
3.0     26416
4.0     27155
5.0     30457
6.0     35160
7.0     45867
8.0     51125
9.0     52084
10.0    48576
11.0    38731
12.0    29128
13.0    39304
14.0     4526
Name: _AGEG5YR, dtype: int64

Résumé de l’IMC brut :


count    452119.000000
mean       2777.583194
std         612.811981
min        1202.000000
25%        2363.000000
50%        2663.000000
75%        3071.000000
max        9769.000000
Name: _BMI5, dtype: float64

In [10]:
donnees_nettoyees = pd.DataFrame({
    "tranche_age": donnees["_AGEG5YR"].where(donnees["_AGEG5YR"] != 14),
    "imc": donnees["_BMI5"] / 100,
    "activite_physique": donnees["EXERANY2"].map({1: 1, 2: 0}),
    "diabete": donnees["diabete"]
})

display(donnees_nettoyees.head())
display(donnees_nettoyees.isna().sum().to_frame("Valeurs manquantes"))
print("Médiane de l'IMC :", donnees_nettoyees["imc"].median())

,tranche_age,imc,activite_physique,diabete
0,9.0,39.16,0.0,0
1,7.0,18.22,1.0,0
2,8.0,27.46,0.0,0
3,9.0,21.97,1.0,0
4,10.0,35.94,0.0,0


,Valeurs manquantes
tranche_age,4526
imc,25620
activite_physique,33094
diabete,0


Médiane de l'IMC : 26.63


In [11]:
lignes_incompletes = donnees_nettoyees.isna().any(axis=1)

nb_incompletes = lignes_incompletes.sum()
nb_total = len(donnees_nettoyees)

print("Lignes incomplètes :", nb_incompletes)
print("Lignes complètes :", nb_total - nb_incompletes)
print(f"Pourcentage de lignes incomplètes : {nb_incompletes / nb_total:.2%}")

Lignes incomplètes : 56310
Lignes complètes : 421429
Pourcentage de lignes incomplètes : 11.79%


In [12]:
donnees_finales = donnees_nettoyees.dropna().reset_index(drop=True)

print("Dimensions finales :", donnees_finales.shape)
print("Valeurs manquantes :", donnees_finales.isna().sum().sum())

print(f"Diabète déclaré avant : {donnees_nettoyees['diabete'].mean():.2%}")
print(f"Diabète déclaré après : {donnees_finales['diabete'].mean():.2%}")

Dimensions finales : (421429, 4)
Valeurs manquantes : 0
Diabète déclaré avant : 13.05%
Diabète déclaré après : 13.13%


In [13]:
from pathlib import Path

fichier_sortie = Path("brfss_2013_nettoye.csv")
donnees_finales.to_csv(fichier_sortie, index=False)

verification = pd.read_csv(fichier_sortie)
assert verification.equals(donnees_finales), "Le CSV relu diffère du tableau final."

print("CSV vérifié :", fichier_sortie.resolve())
print("Dimensions :", verification.shape)

CSV vérifié : C:\Users\Fatim\OneDrive\Desktop\Emsi\Machine learning\projet\brfss_2013_nettoye.csv
Dimensions : (421429, 4)
